# Innerbloom Phase 3: a light model for reading entries

This trains a **small** model (Qwen2.5 1.5B by default) to do one job: read a journal entry and return Innerbloom's JSON (emotion, intensity, summary, tags, themes, people). The goal is to match the 7B model that Innerbloom uses today, while running fast on an ordinary laptop.

**Before you start**
- Runtime > Change runtime type > **A100 GPU**. Check the compute units left in the resource panel (top right).
- Run the cells **one at a time, top to bottom**. Each says what it costs. If anything fails, stop and send the error.
- Everything is saved to your Google Drive, so if Colab disconnects, run the same cell again and it continues.
- Only fictional training data is used. No journal of yours leaves your Mac.
- **When you finish, Runtime > Disconnect and delete runtime**, or it keeps using units.

## 1. Check the GPU (free, seconds)
You should see an **A100** in the table.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Connect Google Drive (free)
A window asks for permission. Results go to `MyDrive/innerbloom/phase3`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/innerbloom/phase3'
import os; os.makedirs(OUT, exist_ok=True); print('saving to', OUT)

## 3. Get the code and libraries (about 2 minutes)
Downloads Innerbloom from GitHub and installs the pinned training libraries.

In [ ]:
BRANCH = 'feature/innerbloom-ai-v2'
!rm -rf /content/innerbloom && git clone -q --depth 1 -b $BRANCH https://github.com/GhanshyamPaunikar/innerbloom /content/innerbloom
%cd /content/innerbloom
!git log --oneline -1
!pip install -q -r ml/requirements-train.txt

## 4. Check everything before spending units (about 1 minute)
Confirms the GPU, the library versions, that the training data passes its checks, and that the longest entry fits. It ends with a line of numbers. If it stops with an error, send it.

In [ ]:
!python -m ml.train_emotion --stage check --out $OUT

## 5. Prepare the training answers (one time, roughly 40-70 minutes, about 6-10 units)
My labels cover emotion, intensity and names. The summary, tags and themes come from the 7B model, so the small model learns the whole format. This runs once and is shared by every size. If it disconnects, run it again: it continues.

In [ ]:
!python -m ml.train_emotion --stage replies --out $OUT

## 6. Smoke test (about 5 minutes, under 1 unit)
Ten training steps, then saves to Drive, reloads, and reads three entries. You should see **SMOKE TEST PASSED**. Do not go on until you do.

In [ ]:
BASE = '1.5b'   # light model; '3b' and '0.5b' are for comparison
!python -m ml.train_emotion --stage smoke --base $BASE --out $OUT

## 7. Train (roughly 15-40 minutes for 1.5B, about 2-6 units)
Up to 3 passes over 1,199 entries, checking the 402 validation entries every 20 steps and stopping early if it stops improving. Saved to Drive as it goes; if it disconnects, run the cell again to resume.

In [ ]:
!python -m ml.train_emotion --stage train --base $BASE --out $OUT

## 8. Compare (roughly 15-30 minutes, about 2-4 units)
Reads the 402 validation entries with the untrained and the trained small model, and compares both with today's 7B. The last lines say whether to keep it: **keep** (better than untrained, nothing worse) and **can_replace_7b_for_reading**.

In [ ]:
!python -m ml.train_emotion --stage evaluate --base $BASE --out $OUT

## 9. Finish
1. Copy everything printed by step 8 and send it back, with the units used (resource panel).
2. **Runtime > Disconnect and delete runtime.**

The trained add-on (`adapter`) stays in Drive. Turning it into a model Ollama can run on your Mac is Phase 6. The locked test set is not used here.